# 10 — Load and visualize saved model results

This notebook is intentionally **read-only with respect to model results**. It needs no firm-level or grid-level source data and never refits a model. Point `RESULT_DIR` at a copied result bundle from notebook 09; figures are always written to the separate `FIG/model_results/` directory.

The model-result directory is read only. Figures are written to the separate `FIG/model_results/` directory. Exports follow a consistent APA 7 presentation: no title inside the plotting area, grayscale-safe encodings, sans-serif text, direct axis labels, and minimal non-data ink. Add the figure number, italicized title, and any explanatory note in the thesis document rather than inside the image.


## Configuration and provenance


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run from the project directory or one of its subdirectories.")


PROJECT_DIR = discover_project_dir()

# Point this path at the result bundle to visualize.
RESULT_DIR = PROJECT_DIR / "ANAL" / "data" / "models"
FIGURE_DIR = PROJECT_DIR / "FIG" / "model_results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = RESULT_DIR / "model_run_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Missing {manifest_path}. Run notebook 09 or copy its complete result bundle.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
display(pd.json_normalize(manifest).T.rename(columns={0: "value"}))


plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

LINE_STYLES = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1))]
MARKERS = ["o", "s", "^", "D", "v", "P", "X"]


## 1. Founding model


In [ ]:
founding = pd.read_csv(RESULT_DIR / "founding_nb2_results.csv", index_col="term")
FOUNDING_LABELS = {
    "log_own_firms": "Lagged firms in cell",
    "log_own_pop": "Population in cell, previous year",
    "log_pop_access_ring_0_15": "Reachable mass, car 0–15 min",
    "log_firms_relative_car_ring_0_15": "Relative firm density, car 0–15 min",
    "log_walk_pop_ring_0_10": "Reachable mass, walk 0–10 min",
    "log_firms_relative_walk_ring_0_10": "Relative firm density, walk 0–10 min",
    "population_growth_yoy": "Year-on-year population growth",
    "log_tt_motorway_exit": "Travel time to motorway exit",
    "walk_pt_routes_10min": "PT routes within 10 min walk",
    "pt_ohne_haltestelle": "No PT stop reachable",
}
plot_terms = [term for term in FOUNDING_LABELS if term in founding.index]
plot_data = founding.loc[plot_terms].iloc[::-1]

fig, ax = plt.subplots(figsize=(8.5, 0.48 * len(plot_data) + 1.6))
positions = np.arange(len(plot_data))
ax.errorbar(
    plot_data["irr"], positions,
    xerr=[plot_data["irr"] - plot_data["irr_ci_lower"], plot_data["irr_ci_upper"] - plot_data["irr"]],
    fmt="o", capsize=3, color="black", markerfacecolor="white",
    markeredgecolor="black", linewidth=1.0,
)
ax.axvline(1, color="0.45", linestyle="--", linewidth=0.8)
ax.set_yticks(positions, [FOUNDING_LABELS[term] for term in plot_data.index])
ax.set_xscale("log")
ax.set_xlabel("Incidence-rate ratio (95% cell-clustered CI; log scale)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "founding_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
all_terms = pd.read_csv(RESULT_DIR / "founding_nb2_all_terms.csv", index_col="term")
periods = all_terms.loc[all_terms.index.str.startswith("period_")].copy()
periods["period"] = periods.index.str.removeprefix("period_")
periods = periods.sort_values("period")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(periods["period"], periods["irr"], marker="o", markersize=3, linewidth=1.0, color="black", markerfacecolor="white")
ax.fill_between(periods["period"], periods["irr_ci_lower"], periods["irr_ci_upper"], color="0.85", linewidth=0)
ax.axhline(1, color="0.45", linestyle="--", linewidth=0.8)
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_ylabel("IRR relative to the first analysis quarter")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "founding_period_effects.png", dpi=300, bbox_inches="tight")
plt.show()

count_fit = pd.read_csv(RESULT_DIR / "founding_nb2_count_fit.csv")
fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(count_fit)); width = 0.38
ax.bar(x - width / 2, count_fit["observed_share"], width, label="Observed", color="white", edgecolor="black", linewidth=0.8, hatch="///")
ax.bar(x + width / 2, count_fit["modelled_share"], width, label="NB2", color="0.60", edgecolor="black", linewidth=0.8)
ax.set_yscale("log"); ax.set_xticks(x, count_fit["births"])
ax.set_xlabel("Births per cell-quarter"); ax.set_ylabel("Share (log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "founding_count_fit.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
sector_path = RESULT_DIR / "founding_sector_population_effects.csv"
if sector_path.exists():
    sector = pd.read_csv(sector_path)
    sector = sector.sort_values("irr")
    if len(sector):
        fig, ax = plt.subplots(figsize=(8, 0.55 * len(sector) + 1.7))
        y = np.arange(len(sector))
        ax.errorbar(sector["irr"], y,
                    xerr=[sector["irr"] - sector["irr_ci_lower"], sector["irr_ci_upper"] - sector["irr"]],
                    fmt="o", capsize=3, color="black", markerfacecolor="white", linewidth=1.0)
        ax.axvline(1, color="0.45", linestyle="--", linewidth=0.8)
        ax.set_yticks(y, sector["sparte_name"])
        ax.set_xscale("log")
        ax.set_xlabel("IRR for reachable population mass (95% CI; log scale)")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "founding_effect_by_sector.png", dpi=300, bbox_inches="tight")
        plt.show()


## 2. Survival model


In [ ]:
survival = pd.read_csv(RESULT_DIR / "survival_cox_results.csv", index_col="term")
SURVIVAL_LABELS = {
    "log_own_pop": "Population in cell",
    "log_own_same": "Same-Fachgruppe firms in cell",
    "log_own_other": "Other firms in cell",
    "log_pop_ring_0_15": "Reachable population mass, 0–15 min",
    "log_same_relative_ring_0_15": "Relative same-group density, 0–15 min",
    "log_other_relative_ring_0_15": "Relative other-group density, 0–15 min",
    "log_tt_motorway_exit": "Travel time to motorway exit",
    "walk_pt_routes_10min": "PT routes within 10 min walk",
    "pt_ohne_haltestelle": "No PT stop reachable",
    "calendar_year": "Calendar-year trend",
}
plot_data = survival.loc[[t for t in SURVIVAL_LABELS if t in survival.index]].iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 0.48 * len(plot_data) + 1.6))
y = np.arange(len(plot_data))
ax.errorbar(plot_data["hazard_ratio"], y,
            xerr=[plot_data["hazard_ratio"] - plot_data["hr_ci_lower"], plot_data["hr_ci_upper"] - plot_data["hazard_ratio"]],
            fmt="o", capsize=3, color="black", markerfacecolor="white", linewidth=1.0)
ax.axvline(1, color="0.45", linestyle="--", linewidth=0.8)
ax.set_yticks(y, [SURVIVAL_LABELS[t] for t in plot_data.index])
ax.set_xscale("log")
ax.set_xlabel("Exit hazard ratio (95% cell-clustered CI; log scale)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "survival_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
ph_screen = pd.read_csv(RESULT_DIR / "survival_schoenfeld_screen.csv").sort_values("spearman_rho")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(ph_screen) + 1.5))
colors = np.where(ph_screen["screen_p_value"] < 0.05, "0.35", "0.80")
ax.barh(np.arange(len(ph_screen)), ph_screen["spearman_rho"], color=colors, edgecolor="black", linewidth=0.6)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(np.arange(len(ph_screen)), [SURVIVAL_LABELS.get(t, t) for t in ph_screen["term"]])
ax.set_xlabel("Spearman correlation: Schoenfeld residual vs event age")
ax.legend(handles=[
    Patch(facecolor="0.35", edgecolor="black", label="Unadjusted p < .05"),
    Patch(facecolor="0.80", edgecolor="black", label="Unadjusted p ≥ .05"),
])
fig.tight_layout()
fig.savefig(FIGURE_DIR / "survival_ph_screen.png", dpi=300, bbox_inches="tight")
plt.show()
display(ph_screen)


In [ ]:
km_path = RESULT_DIR / "survival_km_by_sector.csv"
if km_path.exists():
    km = pd.read_csv(km_path)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for index, ((_, name), group) in enumerate(km.groupby(["sparte", "sparte_name"])):
        group = group.sort_values("time_years")
        ax.step(group["time_years"], group["survival_probability"], where="post", label=name, color="black", linestyle=LINE_STYLES[index % len(LINE_STYLES)], linewidth=1.2)
    ax.set_xlim(left=0); ax.set_ylim(0, 1)
    ax.set_xlabel("Location age (years)"); ax.set_ylabel("Survival probability")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "survival_km_by_sector.png", dpi=300, bbox_inches="tight")
    plt.show()

rates_path = RESULT_DIR / "survival_annual_exit_rates.csv"
if rates_path.exists():
    rates = pd.read_csv(rates_path)
    fig, ax = plt.subplots(figsize=(10, 5))
    for index, ((_, name), group) in enumerate(rates.groupby(["sparte", "sparte_name"])):
        ax.plot(group["year"], group["exit_rate"], marker=MARKERS[index % len(MARKERS)], linestyle=LINE_STYLES[index % len(LINE_STYLES)], color="black", markerfacecolor="white", markersize=4, linewidth=1.0, label=name)
    ax.axvspan(2020, 2021, color="0.90", zorder=0)
    ax.set_xlabel("Calendar year"); ax.set_ylabel("Exits / locations at risk")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "survival_annual_exit_rates.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
sector_path = RESULT_DIR / "survival_sector_effects.csv"
if sector_path.exists():
    sector = pd.read_csv(sector_path).sort_values("hazard_ratio")
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(sector) + 1.7))
    y = np.arange(len(sector))
    ax.errorbar(sector["hazard_ratio"], y,
                xerr=[sector["hazard_ratio"] - sector["hr_ci_lower"], sector["hr_ci_upper"] - sector["hazard_ratio"]],
                fmt="o", capsize=3, color="black", markerfacecolor="white", linewidth=1.0)
    ax.axvline(1, color="0.45", linestyle="--", linewidth=0.8)
    ax.set_yticks(y, sector["sparte_name"])
    ax.set_xscale("log")
    ax.set_xlabel("Hazard ratio for same-group firms in cell (95% CI; log scale)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "survival_effect_by_sector.png", dpi=300, bbox_inches="tight")
    plt.show()

print(f"Figures written to: {FIGURE_DIR}")


## Interpretation guardrails

- In the founding model, the mass/density coefficients are reparameterized. Use `founding_nb2_decomposition.csv` for separate population and firm-stock interpretations.
- A hazard ratio below one means a lower exit hazard (longer survival), not a percentage increase in survival time.
- The proportional-hazards screen is a diagnostic, not an automatic accept/reject rule. Inspect effect size and multiplicity, then estimate a justified time interaction if necessary.
- Sector confidence intervals describe sector-specific effects. Use `founding_sector_joint_test.csv` and `survival_sector_joint_test.csv` for the joint H4 tests; overlapping intervals alone are not tests of equality.
